# Churn Analysis - Telecommunication


<img src="https://media.istockphoto.com/id/1188462768/photo/notepad-with-churn-percent-near-marker-over-desk.jpg?s=612x612&amp;w=0&amp;k=20&amp;c=1cP2pmT6oHcyrAvnMbivVx13jJyw-8Cr2FJos3JblJI=" />


## Team Members
*Idan Arbel - 204344022* 

*Yovel Yefet - 318987815* 

---

## Problem Definition

**What are we predicting?**  
We are predicting whether a telecom customer will **churn** (cancel their subscription) based on their demographics, service usage, and billing information.

**What input data is available?**  
The dataset contains 21 columns and ~7K records describing each customer: demographics (gender, age, dependents), subscribed services (phone, internet, streaming, security), contract details, and billing information. The target label is `Churn` (Yes / No).

**Motivation and applications:**  
Customer churn is a critical business metric for subscription-based companies. Acquiring a new customer costs 5–10× more than retaining an existing one. A reliable churn prediction model enables the business to proactively offer targeted incentives to at-risk customers before they leave.


## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning imports:
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Exploratory Data Analysis

## 1. Load the Dataset

In [ ]:
churn_df = pd.read_csv('churn.csv')
print("Shape:", churn_df.shape)

#### The dataset have 7043 records & 21 columns

#### Show top 5 rows:


In [ ]:
pd.set_option('display.max_columns', None)
churn_df.head(5)

In [ ]:
churn_df.info()

#### Columns
 
Dataset variable description:

<table>
 <tr><th style="text-align: center; background-color: #279CF5; padding: 8px;">Field Name</th><th style="text-align: left; background-color: #279CF5; padding: 8px;">Description</th></tr>
    <tr><th> CustomerId </th><td> String, Should be dropped </td></tr>
    <tr><th> gender </th><td> String, Male or Female. Should be change to Female → 1, Male → 0 </td></tr>
    <tr><th> SeniorCitizen </th><td> Int, Describes if Customer is Senior Citizen (1) or not (0) </td></tr>
    <tr><th> Partner </th><td> String, Should be change to Yes → 1, No → 0  </td></tr>
    <tr><th> Dependents </th><td> String, Should be change to Yes → 1, No → 0  </td></tr>
    <tr><th> tenure </th><td> Int, Number of months the customer has stayed with the company </td></tr>
    <tr><th> PhoneService </th><td> String, Should be change to Yes → 1, No → 0  </td></tr>
    <tr><th> MultipleLines </th><td> String, Needs to be simplified: `No phone service` → `No` <br>Then Should be change to Yes → 1, No → 0 </td></tr>
    <tr><th> InternetService </th><td> String, Multi-Class Categorical <br>One-Hot Encoding conversion (using get_dummies) </td></tr>
    <tr><th> OnlineSecurity </th><td> String, Needs to be simplified: `No internet service` → `No` <br>Then Should be change to Yes → 1, No → 0 </td></tr>
    <tr><th> OnlineBackup </th><td> String, Needs to be simplified: `No internet service` → `No` <br>Then Should be change to Yes → 1, No → 0 </td></tr>
    <tr><th> DeviceProtection </th><td> String, Needs to be simplified: `No internet service` → `No` <br>Then Should be change to Yes → 1, No → 0 </td></tr>
    <tr><th> TechSupport </th><td> String, Needs to be simplified: `No internet service` → `No` <br>Then Should be change to Yes → 1, No → 0 </td></tr>
    <tr><th> StreamingTV </th><td> String, Needs to be simplified: `No internet service` → `No` <br>Then Should be change to Yes → 1, No → 0 </td></tr>
    <tr><th> StreamingMovies </th><td> String, Needs to be simplified: `No internet service` → `No` <br>Then Should be change to Yes → 1, No → 0 </td></tr>
    <tr><th> Contract </th><td> String, Multi-Class Categorical <br>One-Hot Encoding conversion (using get_dummies) </td></tr>
    <tr><th> PaperlessBilling </th><td> String, Should be change to Yes → 1, No → 0  </td></tr>
    <tr><th> PaymentMethod </th><td> String, Multi-Class Categorical <br>One-Hot Encoding conversion (using get_dummies) </td></tr>
    <tr><th> MonthlyCharges </th><td> Float, No further actions needed </td></tr>   
    <tr><th> TotalCharges </th><td> String, Needs conversion to float <br> Manipulate blank values (spaces) and imputate with 'MonthlyCharges' values </td></tr></td></tr>    
    <tr><th> Churn </th><td> Indicator, Should be set to boolean - 0 or 1 </td></tr>
</table>


## 2. Label Distribution

**Hypothesis:** We expect the dataset to be imbalanced - the majority of customers has *been* retained. Understanding the class ratio is essential for choosing the right evaluation metric (accuracy alone can be misleading with imbalanced classes).

In [ ]:
churn_counts = churn_df['Churn'].value_counts()
churn_pct = (churn_df['Churn'].value_counts(normalize=True) * 100).map('{:.2f}%'.format)

print("Churn counts:")
print(churn_counts)
print("------------------------------")
print("% Churn:")
print(churn_pct)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
colors = ['#2196F3', '#F44336']
bars = ax.bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', linewidth=1.5)

# Annotate bars with counts and percentages
for bar, count, pct in zip(bars, churn_counts.values, churn_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 40, f'{count}\n({pct})', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
ax.set_title('Customer Churn Distribution', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Churn', fontsize=10)
ax.set_ylabel('Number of Customers', fontsize=10)
ax.set_ylim(0, churn_counts.max() * 1.3)
plt.show()

## 3. Missing Values Analysis

Before any modelling, we must understand data completeness. Note that `TotalCharges` is stored as an object (string) column (checked in previous prompt above) - this hints at non-numeric entries (spaces) that act as hidden nulls and will require special handling during data preparation.

In [ ]:
churn_df.isnull().sum()

#### No missing Values!
#### But as mentioned above, `TotalCharges` has been defined as *string* in the dataset. Let's check the this attributes values:

In [ ]:
tc_issues = (churn_df['TotalCharges'].str.strip() == '').sum()
tc_issues

#### There are 11 instances in `TotalCharge` <u>that are missing</u> (contains blank space) - In Data Prep section <span style='color:red'> we will imputate</span> these instances.

<br>
<br>
<hr class="dotted">
<br>
<br>

## Before starting with EDA and Visualization - the best practice is to convert the label `Churn` to numeric values.

In [ ]:
# Change the string value in "Churn" to numerical value
churn_df.loc[churn_df.Churn == 'No', 'Churn'] = '0'
churn_df.loc[churn_df.Churn == 'Yes', 'Churn'] = '1'

# Convert "Churn" to int
churn_df = churn_df.astype({"Churn": 'int64'}) 

In [ ]:
# Check new data type:
churn_df['Churn'].dtypes

In [ ]:
customers_cnt = len(churn_df)
customers_churn_cnt = sum(churn_df['Churn'])

# Print the output below using the "print" function
print("The dataset have %d customers and %d of them churned (%.2f%%)" % (customers_cnt, customers_churn_cnt, (customers_churn_cnt/customers_cnt*100)))

<br>
<br>
<hr class="dotted">
<br>
<br>

## 4. Exploratory Data Analysis (EDA)

### 4.1. Gender distribution and churn

In [ ]:
churn_df.groupby('gender').agg({'Churn':['count','sum','mean']}).round(2)

##### No significant difference between the gender groups for churn
<br>Gender mean: <br>
female  -->    	0.27<br>
male    -->    	0.26<br>
both    -->     0.26<br>
<br>
Therfore, almost no impact for this feature

<br>
<br>
<hr class="dotted">
<br>
<br>

### 4.2. Contract distribution and churn
**Hypothesis:** Month-to-month contract customers has the nearest exit points, making them most likely to churn.

In [ ]:
churn_df.groupby('Contract').agg({'Churn':['count','sum','mean']}).round(2)

##### Long-term and short-term contracts differ significantly in terms of churn rate.

<br>Contracts mean: <br>
Monthly Subscription     -->    	0.43<br>
One Year Subscription    -->    	0.11<br>
Two Years Subscription   -->        0.03<br>
<br>
Conclusion: The longer the contract period --> the chance for churn decreases

In [ ]:
plt.figure(figsize=(8, 4))
colors = ['#2196F3', '#F44336']

ax = sns.countplot(data=churn_df, x='Contract', hue='Churn', palette=colors)

plt.title('Churn Risk by Contract Type', fontsize=14, fontweight='bold')
plt.ylabel('Number of Customers')
plt.xlabel('Contract Type')
plt.legend(title='Churn', loc='upper right')

plt.show()

<br>
<br>
<hr class="dotted">
<br>
<br>

### 4.3. Customers tenure and churn
**Hypothesis:** Customers who considered as new (shorter tenure) are more likely to churn - they have not yet built loyalty or sunk cost with the provider.

In [ ]:
plt.figure(figsize=(8, 4))

sns.kdeplot(data=churn_df[churn_df['Churn'] == 0]['tenure'], label='Retained', fill=True, color='#2196F3')
sns.kdeplot(data=churn_df[churn_df['Churn'] == 1]['tenure'], label='Churned', fill=True, color='#F44336')
plt.title('Customer Churn Risk Over Time (Tenure)', fontsize=14, fontweight='bold')
plt.ylabel('Density (Concentration of Customers)')
plt.xlabel('Tenure (Months with the company)')
plt.legend()
plt.xlim(left=0)

plt.show()

#### As it might have been expected, most of churn comes out of "new customers" (tenure < 12 months).
#### Let's standerize and make a categorial attribute that assign 


In [ ]:
churn_df['tenureGroup'] = None
churn_df.loc[(churn_df['tenure'] <= 12),'tenureGroup'] = '0-12 Months'
churn_df.loc[(churn_df['tenure'] > 12) & (churn_df['tenure'] <= 24),'tenureGroup'] = '13-24 Months'
churn_df.loc[(churn_df['tenure'] > 24) & (churn_df['tenure'] <= 36),'tenureGroup'] = '25-36 Months'
churn_df.loc[(churn_df['tenure'] > 36),'tenureGroup']= '37+ Months'


In [ ]:
churn_df.groupby('tenureGroup').agg({'Churn':['count','sum','mean']}).round(2)

##### Customer maturity groups to emphasize the difference between customers groups for churn
<br>tenureGroup mean: <br>
0-12 Months     -->    	0.47<br>
13-24 Months    -->    	0.29<br>
25-36 Months    -->    	0.22<br>
37+ Months    -->    	0.12<br>
<br>
As long as the tenurity of a customer is lower --> the chance for churn increases

In [ ]:
tenure_data = churn_df.groupby('tenureGroup')['Churn'].mean() * 100

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(tenure_data.index, tenure_data.values, 
              color=sns.color_palette('flare', len(tenure_data)))

ax.bar_label(bars, fmt='%.1f%%', padding=3)

ax.set_title('Churn Rate Distribution by Tenure Group', fontsize=14, fontweight='bold')
ax.set_xlabel('Tenure Group', fontsize=12)
ax.set_ylabel('Churn Rate (%)', fontsize=12)
sns.despine()

plt.tight_layout()
plt.show()

<br>
<br>
<hr class="dotted">
<br>
<br>

### 4.4. Customers seniority and churn
**Hypothesis:** senior citizens might be more price-sensitive or less digitally engaged.

In [ ]:
churn_df.groupby('SeniorCitizen').agg({'Churn':['count','sum','mean']}).round(2)

##### Senior Citizens are likely to churn

<br>SeniorCitizen mean: <br>
Non-Senior Citizenship --> 0.24<br>
Senior Citizenship --> 0.42<br>
<br>
For customers categorized as seniors --> the chance for churn increases


In [ ]:
senior_data = churn_df.groupby(['SeniorCitizen', 'Churn']).size().unstack(fill_value=0)
senior_data.index = ['Non-Senior', 'Senior']

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

colors = ['#2196F3', '#F44336']
labels = ['Retained', 'Churned']

axes[0].pie(senior_data.iloc[0], labels=labels, autopct='%1.1f%%', startangle=90, colors=colors, wedgeprops={'edgecolor': 'white'})
axes[0].set_title('Non-Senior Citizen Churn', fontweight='bold')

axes[1].pie(senior_data.iloc[1], labels=labels, autopct='%1.1f%%', startangle=90, colors=colors, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Senior Citizen Churn', fontweight='bold')

plt.tight_layout()
plt.show()


### 4.5. Payment methods and churn
**Hypothesis 1:** Automatic payment methods draw less attention from customers and therefore might ***reduce*** the churn rate. 
<br>
**Hypothesis 2:** Customers paying by electronic check may indicate less commitment and ***higher*** churn risk.

In [ ]:
churn_df.groupby('PaymentMethod').agg({'Churn':['count','sum','mean']}).round(2)

##### Analyzing Churn Differences Among Customer Groups by Payment Method
<br>PaymentMethod mean: <br>
Bank transfer (automatic)    -->    	0.17<br>
Credit card (automatic)    -->    	0.15<br>
Electronic Check    -->    	0.45<br>
Mailed Check   -->    	0.19<br>
<br>


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

payment_churn_rate = churn_df.groupby('PaymentMethod')['Churn'].apply(lambda x: (x == 1).mean() * 100).sort_values(ascending=False)

bars = ax.bar(payment_churn_rate.index, payment_churn_rate.values,
              color=sns.color_palette('magma', len(payment_churn_rate)),
              edgecolor='white', linewidth=1.2)

ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=10)

ax.set_title('Churn Rate by Payment Method', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Payment Method', fontsize=11)
ax.set_ylabel('Churn Rate (%)', fontsize=11)
ax.set_ylim(0, payment_churn_rate.max() * 1.2)

ax.tick_params(axis='x')
sns.despine()

plt.tight_layout()
plt.show()


### 4.6. Monthly Charges and churn

In [ ]:
churn_df.groupby('Churn').agg({'MonthlyCharges':['count','mean','median','std']}).round(2)

##### Analyzing the relationship between Monthly Charges and Churn

**MonthlyCharges mean by Churn:**
- Retained (0): ~&#36;61.27
- Churned&nbsp;&nbsp;(1): ~&#36;74.44

Churned customers pay on average ~22% more per month, with their distribution skewed toward the higher end of the price range.  This suggests perceived value-for-money is a key driver of churn.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE - distribution comparison
sns.kdeplot(data=churn_df[churn_df['Churn'] == 0]['MonthlyCharges'],
            label='Retained', fill=True, color='#2196F3', ax=axes[0])
sns.kdeplot(data=churn_df[churn_df['Churn'] == 1]['MonthlyCharges'],
            label='Churned',  fill=True, color='#F44336', ax=axes[0])
axes[0].set_title('Monthly Charges Distribution by Churn', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].set_ylabel('Density (Concentration of Customers)')
axes[0].legend()

# Boxplot - quartiles, median, outliers
sns.boxplot(data=churn_df, x='Churn', y='MonthlyCharges',
            hue='Churn', palette=['#2196F3', '#F44336'],
            legend=False, ax=axes[1])
axes[1].set_title('Monthly Charges Spread by Churn', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Customer Status')
axes[1].set_ylabel('Monthly Charges ($)')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Retained', 'Churned'])

# Annotate the means on the boxplot
means = churn_df.groupby('Churn')['MonthlyCharges'].mean()
for i, m in enumerate(means):
    axes[1].scatter(i, m, marker='D', color='white', edgecolor='black', s=60, zorder=5)
    axes[1].text(i, m + 2, f'${m:.2f}', ha='center', fontsize=10, fontweight='bold')

sns.despine()
plt.tight_layout()
plt.show()

### 4.7. Internet Service and churn
**Hypothesis:** Some Internet Services tend to churn, due to various reasons (product/support) 

In [ ]:
churn_df.groupby('InternetService').agg({'Churn':['count','sum','mean']}).round(2)

##### Analyzing Churn Differences Among Customer's Internet type service groups
<br>InternetService mean: <br>
DSL    -->    	0.19<br>
Fiber optic    -->    	0.42<br>
No    -->    	0.07<br>
<br>


In [ ]:
# Calculate metrics
internet_churn = churn_df.groupby('InternetService').agg(total_customers=('Churn', 'count'), churn_rate=('Churn', 'mean'))
    # .reset_index()

# Convert churn_rate to percentage
internet_churn['churn_rate'] = internet_churn['churn_rate'] * 100

# Create figure and primary axis
fig, ax1 = plt.subplots(figsize=(8 ,6))

# Plot total customers (Bar Chart) on primary axis
sns.barplot(
    data=internet_churn, 
    x='InternetService', 
    y='total_customers', 
    color='#2196F3', 
    ax=ax1, 
    label='Total Customers'
)

# Set labels for primary axis
ax1.set_xlabel('Internet Service', fontsize=12)
ax1.set_ylabel('Total Customers', fontsize=12, color='#2196F3')
ax1.tick_params(axis='y', labelcolor='#2196F3')
ax1.set_ylim(0, internet_churn['total_customers'].max() * 1.2)

# Create secondary axis
ax2 = ax1.twinx()

# Plot churn rate (Line Chart) on secondary axis
sns.lineplot(
    data=internet_churn, 
    x='InternetService', 
    y='churn_rate', 
    color='#F44336', 
    marker='o', 
    linewidth=2.5, 
    markersize=8, 
    ax=ax2, 
    label='Churn Rate (%)'
)

# Set labels for secondary axis
ax2.set_ylabel('Churn Rate (%)', fontsize=12, color='#F44336')
ax2.tick_params(axis='y', labelcolor='#F44336')
ax2.set_ylim(0, internet_churn['churn_rate'].max() * 1.4)

# Add values on top of the bars and line points
for i, row in internet_churn.iterrows():
    # Bar values
    ax1.text(i, row['total_customers'] + 50, f"{int(row['total_customers'])}", ha='center', color='#2196F3', fontweight='bold')
    # Line values
    ax2.text(i, row['churn_rate'] + 1, f"{row['churn_rate']:.1f}%", ha='center', color='#C62828', fontweight='bold')
plt.title('Total Customers and Churn Rate by Internet Service', fontsize=14, fontweight='bold', pad=15)


lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper right')
ax2.get_legend().remove()

plt.tight_layout()
plt.show()

## 5. EDA findings

| Finding | Evidence |
|---------|----------|
| **~26% of customers churn** | Imbalanced classes - accuracy alone is insufficient as the sole metric |
| **Gender has almost no effect** | Churn rates nearly identical for male and female customers |
| **Senior citizens churn more** | ~42% churn rate vs. ~24% for non-seniors |
| **Churn rate tend to lower overtime** | Decline trend in Churn rate due to customer tenure|
| **Month-to-month contracts drive churn** | ~43% churn vs. ~11% (one year) and ~3% (two year) |
| **Electronic check → higher churn** | Payment method correlates with commitment level |
| **Higher monthly charges → more churn** | Churned customers pay significantly more per month |
| **Fiber Optic service → higher churn** | ~42% churn vs. ~19% (DSL) and ~7% (No internet services) |



# Data Preparation

## Goals
This part transforms the raw churn dataset into a clean, fully numeric feature matrix ready for machine learning.

**Steps:**
1. Load & inspect raw data
2. Remove non-informative features
3. Encode the target variable
4. Fix TotalCharges (hidden nulls)
5. Encode binary Yes/No columns
6. Simplify and encode multi-category service columns
7. One-hot encode remaining categorical columns
8. Correlation analysis with the target (label)
9. Final validation

## 1. Load the Dataset

In [ ]:
churn_df = pd.read_csv('churn.csv')
print("Shape:", churn_df.shape)

In [ ]:
churn_df.head(5)

In [ ]:
churn_df.info()

## 2a. Feature Removal - customerID


In [ ]:
print("Unique customerID values:", churn_df['customerID'].nunique())
print("Total rows:", len(churn_df))
print(f"Uniqueness: {churn_df['customerID'].nunique() / len(churn_df) * 100:.1f}%")

**Decision:** Drop `customerID`.

**Rationale:** `customerID` is a unique identifier assigned to each customer. It has 100% unique values across the dataset and carries no information about customer behaviour or demographics. <br> Including it would only add noise.

In [ ]:
churn_df = churn_df.drop(columns=['customerID'])
print("\nShape after dropping customerID:", churn_df.shape)

## 2b. Feature Removal — `gender`

**Decision:** Drop `gender`.

**Rationale:** EDA Section 4.1 showed that Male and Female customers churn at identical rates (~26% each). <br>The column carries no predictive signal and adding noise-only features can hurt model accuracy by diluting the importance scores of genuinely predictive features.

In [ ]:
churn_df = churn_df.drop(columns=['gender'])
print("Shape after dropping gender:", churn_df.shape)


In [ ]:
# Show all remainng columns after dropping:
print("Remaining columns:", list(churn_df.columns))

## 3. Target Encoding - Churn (Label)

ML models require numeric labels. We map `Yes → 1` and `No → 0`.


In [ ]:
# Change the string value in "Churn" to numerical value
churn_df.loc[churn_df.Churn == 'No', 'Churn'] = '0'
churn_df.loc[churn_df.Churn == 'Yes', 'Churn'] = '1'

# Convert "Churn" to int
churn_df = churn_df.astype({"Churn": 'int64'}) 

In [ ]:
print("Encoded Churn distribution:")
print(churn_df['Churn'].value_counts())

## 4. Complete TotalCharges Values - Hidden Nulls

The `TotalCharges` column is stored as `object` (string). Some entries are whitespace-only strings, which represent customers with 0 tenure (new customers whose total charge is undefined). We decided to convert those instances to numeric - whitespace entries become `NaN` - then fill with the `MonthlyCharges`.

In [ ]:
# Check whitespace:
whitespace_mask = churn_df['TotalCharges'].str.strip() == ''
print(f"Whitespace-only entries in TotalCharges: {whitespace_mask.sum()}")

In [ ]:
# Convert to numeric, invalid parsing will be set as NaN
churn_df['TotalCharges'] = pd.to_numeric(churn_df['TotalCharges'], errors='coerce')

# Fill the resulting NaN values with the 'MonthlyCharges' of the same record
churn_df['TotalCharges'] = churn_df['TotalCharges'].fillna(churn_df['MonthlyCharges'])

# Verify that there are no more NaNs
print(f"NaN entries after fill: {churn_df['TotalCharges'].isnull().sum()}")

## 5. Binary Column Encoding

Several columns are straightforward binary Yes/No. We encode these as `1/0` integers directly.

| Column | Encoding |
|--------|----------|
| `Partner` | Yes → 1, No → 0 |
| `Dependents` | Yes → 1, No → 0 |
| `PhoneService` | Yes → 1, No → 0 |
| `PaperlessBilling` | Yes → 1, No → 0 |

In [ ]:
# Converting binary columns to 1/0:
binary_yes_no_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

for col in binary_yes_no_cols:
    churn_df[col] = (churn_df[col] == 'Yes').astype(int)

In [ ]:
# Convertion check sample:
churn_df[['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']].head()

## 6a. Service Add-on Columns - Simplify Multi-Category Values

The internet and phone service add-on columns (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `MultipleLines`) contain three values: `Yes`, `No`, and `No internet service` / `No phone service`.

**Decision:** Collapse `"No internet service"` and `"No phone service"` into `"No"`, then encode `Yes → 1`, `No → 0`.

**Rationale:** A customer without internet service inherently has no online add-ons - this is equivalent to `No`. Keeping a third category would create redundancy with the `InternetService` column.

In [ ]:
# Converting multi-categorial columns to 1/0:

multi_category_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                        'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in multi_category_cols:
    print(f"{col}: {churn_df[col].unique()}")
    churn_df[col] = churn_df[col].replace({'No internet service': 'No', 'No phone service': 'No'})
    churn_df[col] = (churn_df[col] == 'Yes').astype(int)

In [ ]:
# Convertion check sample:
churn_df[multi_category_cols].head()

## 6b. Feature Engineering — `services_count` & `TenureGroup`

Two new features derived from existing columns to give the models stronger numeric signals:

**`services_count`** — total number of add-on services a customer subscribes to (range 0–7). The EDA showed customers with more services have a higher switching cost and churn less.<br> Summing the 7 already-encoded binary service columns collapses that pattern into one numeric feature.

**`TenureGroup`** — groups tenure into 4 bands matching the EDA (Section 4.3): 0 = 0–12 months, 1 = 13–24 months, 2 = 25–36 months, 3 = 37+ months. Each band showed a different churn rate in the EDA, so using the same grouping here keeps the analysis consistent.

In [ ]:
# Adding services_count: total add-on services subscribed per customer:

service_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup','DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
churn_df['services_count'] = churn_df[service_cols].sum(axis=1)


# Adding TenureGroup: 4 bins used in EDA (Section 4.3)
# Dictionary: Group 0: 0-12m | Group 1: 13-24m | Group 2: 25-36m | Group 3: 37+m
churn_df['TenureGroup'] = pd.cut(churn_df['tenure'],bins=[-1, 12, 24, 36, 72],labels=[0, 1, 2, 3]).astype(int)

print("services_count distribution:")
print(churn_df['services_count'].value_counts().sort_index())
print("----------------------------")
print(f'TenureGroup NaNs: {churn_df["TenureGroup"].isna().sum()}')
print("----------------------------")
print(churn_df[['tenure', 'TenureGroup', 'services_count']].head(10))

# Drop original tenure — TenureGroup captures the same information as an ordinal feature.
# Keeping both would create redundancy and hurt KNN distance calculations.
churn_df = churn_df.drop(columns=['tenure'])
print("Shape after dropping tenure:", churn_df.shape)

## 7. One-Hot Encoding - Multi-Class Categorical Columns

3 columns have more than two unordered categories and cannot be label-encoded without implying an ordinal relationship. We used `pd.get_dummies()` to create binary indicator columns.

| Column | Categories |
|--------|------------|
| `InternetService` | DSL, Fiber optic, No |
| `Contract` | Month-to-month, One year, Two year |
| `PaymentMethod` | Electronic check, Mailed check, Bank transfer (automatic), Credit card (automatic) |


In [ ]:
churn_df = pd.get_dummies(churn_df, columns=['InternetService'])
churn_df = pd.get_dummies(churn_df, columns=['Contract'])
churn_df = pd.get_dummies(churn_df, columns=['PaymentMethod'])
churn_df = pd.get_dummies(churn_df, columns=['TenureGroup'])

churn_df.head(10)

In [ ]:
churn_df.shape

In [ ]:
# Converting Boolean to integers:

churn_df['InternetService_DSL'] = churn_df['InternetService_DSL'].map({False:0, True:1})
churn_df['InternetService_Fiber optic'] = churn_df['InternetService_Fiber optic'].map({False:0, True:1})
churn_df['InternetService_No'] = churn_df['InternetService_No'].map({False:0, True:1})

churn_df['Contract_Month-to-month'] = churn_df['Contract_Month-to-month'].map({False:0, True:1})
churn_df['Contract_One year'] = churn_df['Contract_One year'].map({False:0, True:1})
churn_df['Contract_Two year'] = churn_df['Contract_Two year'].map({False:0, True:1})

churn_df['PaymentMethod_Bank transfer (automatic)'] = churn_df['PaymentMethod_Bank transfer (automatic)'].map({False:0, True:1})
churn_df['PaymentMethod_Credit card (automatic)'] = churn_df['PaymentMethod_Credit card (automatic)'].map({False:0, True:1})
churn_df['PaymentMethod_Electronic check'] = churn_df['PaymentMethod_Electronic check'].map({False:0, True:1})
churn_df['PaymentMethod_Mailed check'] = churn_df['PaymentMethod_Mailed check'].map({False:0, True:1})


churn_df['TenureGroup_0'] = churn_df['TenureGroup_0'].map({False:0, True:1})
churn_df['TenureGroup_1'] = churn_df['TenureGroup_1'].map({False:0, True:1})
churn_df['TenureGroup_2'] = churn_df['TenureGroup_2'].map({False:0, True:1})
churn_df['TenureGroup_3'] = churn_df['TenureGroup_3'].map({False:0, True:1})

churn_df.head(10)

In [ ]:
# New shape of the dataset - after converting to dummies:

print("Shape after one-hot encoding:", churn_df.shape)
print("New columns:", list(churn_df.columns))

## 8. Convert DataFrame Columns to Float

Ensuring all columns are floats for compatibility with scikit-learn estimators.

In [ ]:
churn_df = churn_df.astype(float)
print("Any remaining nulls:", churn_df.isnull().sum().sum())
print("Dtypes:")
print(churn_df.dtypes)


## 9. Correlation Analysis with Churn

We compute the Pearson correlation of each feature with the `Churn` target. Features with high absolute correlation are likely the most predictive.

Note that correlation only captures linear relationships, so low correlation does not necessarily mean a feature is useless for tree-based models.

In [ ]:
# Ordering the 'Churn' (The label) to be the first column in DataFrame

label_col = churn_df.pop('Churn')
churn_df.insert(0, 'Churn', label_col)


In [ ]:
# Correlation in tabluar display:

churn_corr = churn_df.corr()
churn_corr

In [ ]:
# Plotting Heatmap for correlation:

plt.figure(figsize=(16, 10))
sns.heatmap(churn_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.show()

## 10. Final Data Validation

We confirm the cleaned dataset has no missing values, all numeric types, and the expected shape before saving it for Machine Learning models

In [ ]:
print("=" * 40)
print("FINAL DATASET SUMMARY")
print("=" * 40)
print(f"Shape: {churn_df.shape}")
print(f"Missing values: {churn_df.isnull().sum().sum()}")
print(f"All numeric: {churn_df.dtypes.apply(lambda x: np.issubdtype(x, np.number)).all()}")
print("=" * 40)
churn_df.head()


## 11. Data Preparation Summary

| Step | Transformation | Columns Affected |
|------|---------------|------------------|
| Drop unique identifier | Removed `customerID` | 1 column dropped |
| Drop non-predictive feature | Removed `gender` (identical churn rate across groups) | 1 column dropped |
| Target encoding | `Churn`: Yes→1, No→0 | `Churn` |
| Fix hidden nulls | `TotalCharges`: spaces→NaN→`MonthlyCharges` | `TotalCharges` |
| Binary encoding | Yes→1, No→0 | `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling` |
| Simplify service cols | Collapse 3-category → binary, then encode | 7 service columns |
| One-hot encoding | `pd.get_dummies` | `InternetService`, `Contract`, `PaymentMethod`, `TenureGroup` |
| Feature engineering | `services_count`: sum of 7 binary service cols (0–7) | 1 new column |
| Feature engineering | `TenureGroup`: ordinal tenure bins 0–3 (kept as integer — ordering matters) | 1 new column |
| Drop redundant feature | Removed `tenure` (replaced by ordinal `TenureGroup`) | 1 column dropped |
| Type conversion | All columns → float64 | All columns |

**Result:** Raw 21-column dataset → clean **numeric matrix** with **29 features (+1 label)** ready for machine learning.

In [ ]:
# Saving dataset to CSV

churn_df.to_csv('churn_df_cleaned.csv')

# Machine Learning Models

Modeling Hypothesis

We hypothesize that a combination of contract type, tenure, and monthly charges will be the strongest predictors of churn. Tree-based methods (Decision Tree, Random Forest) should outperform KNN because churn patterns are likely rule-based (e.g., "month-to-month + high charges → churn") rather than distance-based.

We test three algorithms with hyperparameter tuning and compare their best configurations.

---


<p style="color:#FF0000;font-size:20px">
    <b>
        <u>We are about to load the original dataset again.</u>
    </b>
        <br>
        This time we will keep the *customerID* attribute for further actions.
        </u>
    </b>
</p>

In [ ]:
churn_df = pd.read_csv('churn.csv')

print(churn_df.shape)
churn_df.head(5)

## 1.Data Preparation

In [ ]:
# Target Encoding - Churn (Label):
churn_df.loc[churn_df.Churn == 'No', 'Churn'] = '0'
churn_df.loc[churn_df.Churn == 'Yes', 'Churn'] = '1'

# Complete TotalCharges Values - Hidden Nulls:
churn_df['TotalCharges'] = pd.to_numeric(churn_df['TotalCharges'], errors='coerce')
churn_df['TotalCharges'] = churn_df['TotalCharges'].fillna(churn_df['MonthlyCharges'])

# Drop non-predictive column (EDA Section 4.1: identical churn rate for Male/Female):
churn_df = churn_df.drop(columns=['gender'])

# Binary Column Encoding:
binary_yes_no_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_yes_no_cols:
    churn_df[col] = (churn_df[col] == 'Yes').astype(int)

# Service Add-on Columns - Simplify Multi-Category Values:
multi_category_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in multi_category_cols:
    churn_df[col] = churn_df[col].replace({'No internet service': 'No', 'No phone service': 'No'})
    churn_df[col] = (churn_df[col] == 'Yes').astype(int)

# Adding services_count: total add-on services subscribed per customer
service_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup','DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
churn_df['services_count'] = churn_df[service_cols].sum(axis=1)


# Adding TenureGroup: 
churn_df['TenureGroup'] = pd.cut(churn_df['tenure'],bins=[-1, 12, 24, 36, 72],labels=[0, 1, 2, 3]).astype(int)

# One-Hot Encoding - Multi-Class Categorical Columns:
churn_df = pd.get_dummies(churn_df, columns=['InternetService'])
churn_df = pd.get_dummies(churn_df, columns=['Contract'])
churn_df = pd.get_dummies(churn_df, columns=['PaymentMethod'])
churn_df = pd.get_dummies(churn_df, columns=['TenureGroup'])

churn_df['InternetService_DSL'] = churn_df['InternetService_DSL'].map({False:0, True:1})
churn_df['InternetService_Fiber optic'] = churn_df['InternetService_Fiber optic'].map({False:0, True:1})
churn_df['InternetService_No'] = churn_df['InternetService_No'].map({False:0, True:1})

churn_df['Contract_Month-to-month'] = churn_df['Contract_Month-to-month'].map({False:0, True:1})
churn_df['Contract_One year'] = churn_df['Contract_One year'].map({False:0, True:1})
churn_df['Contract_Two year'] = churn_df['Contract_Two year'].map({False:0, True:1})

churn_df['PaymentMethod_Bank transfer (automatic)'] = churn_df['PaymentMethod_Bank transfer (automatic)'].map({False:0, True:1})
churn_df['PaymentMethod_Credit card (automatic)'] = churn_df['PaymentMethod_Credit card (automatic)'].map({False:0, True:1})
churn_df['PaymentMethod_Electronic check'] = churn_df['PaymentMethod_Electronic check'].map({False:0, True:1})
churn_df['PaymentMethod_Mailed check'] = churn_df['PaymentMethod_Mailed check'].map({False:0, True:1})

churn_df['TenureGroup_0'] = churn_df['TenureGroup_0'].map({False:0, True:1})
churn_df['TenureGroup_1'] = churn_df['TenureGroup_1'].map({False:0, True:1})
churn_df['TenureGroup_2'] = churn_df['TenureGroup_2'].map({False:0, True:1})
churn_df['TenureGroup_3'] = churn_df['TenureGroup_3'].map({False:0, True:1})

# Drop original tenure (replaced by ordinal TenureGroup):
churn_df = churn_df.drop(columns=['tenure'])

# Converting DataFrame Columns to Float (except customerID):
cols_to_convert = churn_df.columns.difference(['customerID'])
churn_df[cols_to_convert] = churn_df[cols_to_convert].astype(float)

In [ ]:
churn_df.head()

In [ ]:
# Check Nulls and Data types
print(churn_df.isna().sum())
print("-" * 60)
churn_df.dtypes

## 2. Train-Test Split

We split the dataset into **training** (80%) and **test** (20%) sets.
The model learns from training data and is evaluated on unseen test data to assess generalisation.
We drop `customerID` (non-informative identifier) and `Churn` (target label) before feature extraction.

**Why stratified splitting?**
The dataset is imbalanced: 73.5% No-churn vs 26.5% Yes-churn. Without stratification, a random
shuffle can place a slightly different class ratio into train and test — making accuracy scores less
stable and slightly pessimistic. Passing `stratify=churn_df[label]` forces scikit-learn to preserve
the original 26.5% churn rate in both splits. This ensures the model is trained and evaluated on
representative samples of the actual class distribution.

In [ ]:
label  = 'Churn'
id_col = 'customerID'

test_size = int(len(churn_df) * 0.2)  # 20% of dataset

train, test = train_test_split(
    churn_df,
    test_size=test_size,
    random_state=0,
    shuffle=True,
    stratify=churn_df[label]   # preserve 73.5/26.5 class ratio in both sets
)

x_train = train.drop([label, id_col], axis=1)
y_train = train[label]
cust_train = train[id_col]

x_test = test.drop([label, id_col], axis=1)
y_test = test[label]
cust_test = test[id_col]

print(f'Train size : {x_train.shape}')
print(f'Test size  : {x_test.shape}')
print(f'Train churn rate: {y_train.astype(float).mean():.3f}')
print(f'Test  churn rate: {y_test.astype(float).mean():.3f}')

## 3. Model No.1: Decision Tree

**Hypothesis:** Churn follows rule-based patterns,making Decision Trees a natural fit.  
We start with `max_depth=3` for interpretability - a shallow tree that captures the most important splits without overfitting.

In [ ]:
clf_dt = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_dt.fit(x_train, y_train)

y_test_pred_dt  = clf_dt.predict(x_test)
y_train_pred_dt = clf_dt.predict(x_train)

dt_test_acc  = accuracy_score(y_test,  y_test_pred_dt)
dt_train_acc = accuracy_score(y_train, y_train_pred_dt)

output_dt = pd.DataFrame({
    'customerID':      cust_test.values,
    'churn_actual':    y_test.values,
    'churn_predicted': y_test_pred_dt
})

print(f'Decision Tree  -  Train Accuracy: {dt_train_acc:.4f}  |  Test Accuracy: {dt_test_acc:.4f}')
output_dt.head()

### Decision Tree Visualisation

Visualising the learned tree reveals which features the model splits on.  
The depth-3 tree is interpretable: we can trace the decision path from root to leaf.

In [ ]:
plt.figure(figsize=(22, 8))
plot_tree(
    clf_dt,
    feature_names=x_train.columns,class_names=['No Churn', 'Churn'],filled=True, rounded=True, fontsize=8)
plt.title('Decision Tree (max_depth=3) - Churn Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Model No.2: Random Forest

Random Forest builds multiple decision trees and averages their predictions,  
reducing variance and overfitting compared to a single tree.  
We use `n_estimators=100` trees and `max_depth=3` as a baseline - consistent with the Decision Tree for a fair initial comparison.

In [ ]:
clf_rf = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=1)
clf_rf.fit(x_train, y_train)

y_test_pred_rf  = clf_rf.predict(x_test)
y_train_pred_rf = clf_rf.predict(x_train)

rf_test_acc  = accuracy_score(y_test,  y_test_pred_rf)
rf_train_acc = accuracy_score(y_train, y_train_pred_rf)

output_rf = pd.DataFrame({
    'customerID':      cust_test.values,
    'churn_actual':    y_test.values,
    'churn_predicted': y_test_pred_rf
})

print(f'Random Forest - Train Accuracy: {rf_train_acc:.4f}  |  Test Accuracy: {rf_test_acc:.4f}')
output_rf.head()

### Random Forest Feature Importance

Feature importance measures how much each feature reduces impurity across all trees.  
This gives an interpretable view of the strongest churn predictors- expected to align with our EDA findings.

In [ ]:
feature_importances = clf_rf.feature_importances_
features = x_train.columns

stats = pd.DataFrame({'Feature': features, 'Importance': feature_importances})
stats_sorted = stats.sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(stats_sorted['Feature'], stats_sorted['Importance'], color='steelblue', edgecolor='white')
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Random Forest - Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

stats.sort_values('Importance', ascending=False).reset_index(drop=True)

## 5. K-Nearest Neighbors (KNN)

KNN classifies a sample by majority vote of its k nearest neighbours in feature space.  
Churn prediction is more rule-based than distance-based, so KNN may be less suited - but we test it for completeness.  
We start with `n_neighbors=3` without scaling as a baseline.

In [ ]:
clf_knn = KNeighborsClassifier(n_neighbors=3)
clf_knn.fit(x_train, y_train)

y_test_pred_knn  = clf_knn.predict(x_test)
y_train_pred_knn = clf_knn.predict(x_train)

knn_test_acc  = accuracy_score(y_test,  y_test_pred_knn)
knn_train_acc = accuracy_score(y_train, y_train_pred_knn)

output_knn = pd.DataFrame({
    'customerID':      cust_test.values,
    'churn_actual':    y_test.values,
    'churn_predicted': y_test_pred_knn
})

print(f'KNN (k=3)  -  Train Accuracy: {knn_train_acc:.4f}  |  Test Accuracy: {knn_test_acc:.4f}')
output_knn.head()

## 6. Interim Conclusions - Model Comparison

Comparing baseline test accuracy across all three algorithms before overfitting analysis.

In [ ]:
summary = pd.DataFrame({
    'Algorithm':       ['Decision Tree (depth=3)', 'Random Forest (100 trees, depth=3)', 'KNN (k=3, no scaling)'],
    'Train Accuracy':  [dt_train_acc,  rf_train_acc,  knn_train_acc],
    'Test Accuracy':   [dt_test_acc,   rf_test_acc,   knn_test_acc]
})
summary = summary.sort_values('Test Accuracy', ascending=False).reset_index(drop=True)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#1f77b4', '#aec7e8']

summary.set_index('Algorithm').plot(kind='bar', ax=ax, width=0.7, color=colors)
ax.set_title('Model Performance Comparison', fontsize=14)
ax.set_ylabel('Accuracy Score', fontsize=12)
ax.set_xlabel('Algorithm', fontsize=12)
ax.set_ylim(0.70, 1.0) 

ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.set_xticklabels(ax.get_xticklabels(), rotation=10, ha='right')

# Loop through the bars to annotate the exact score on top of each bar
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f"{p.get_height():.3f}", 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='bottom', 
                    xytext=(0, 5), 
                    textcoords='offset points', 
                    fontsize=9)

plt.show()

### Summary:
#### **Winner Model** (Based accuracy, yet tuned) : `KNN`
#### `Decision Tree` & `Random Forest` seems to have the same **accuracy** results.

---
# Overfitting Analysis

Overfitting occurs when a model performs well on training data but poorly on unseen test data.  
We investigate the overfitting behaviour of each algorithm by varying their key hyperparameters  
and comparing **train** vs. **test** accuracy.

## 7. Decision Tree - Overfitting Analysis

We vary `max_depth` from 1 to 20.  
As depth increases the tree memorises training data (overfits) and the gap between train and test accuracy widens.  
The optimal depth is where test accuracy peaks before declining.

In [ ]:
l_depth = []
l_dt_test = []
l_dt_train = []

for depth in range(1, 21):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(x_train, y_train)    
    l_dt_test.append(accuracy_score(y_test,  clf.predict(x_test)))
    l_dt_train.append(accuracy_score(y_train, clf.predict(x_train)))
    l_depth.append(depth)

best_dt_depth = l_depth[l_dt_test.index(max(l_dt_test))]
best_dt_acc   = max(l_dt_test)

plt.figure(figsize=(8, 4))
plt.plot(l_depth, l_dt_train, marker='o', label='Train Accuracy', color='royalblue')
plt.plot(l_depth, l_dt_test,  marker='s', label='Test Accuracy',  color='tomato')
plt.axvline(best_dt_depth, color='gray', linestyle='--', alpha=0.7, label=f'Best depth={best_dt_depth}')
plt.xlabel('max_depth', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Decision Tree - Overfitting: Train vs Test Accuracy', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Best Decision Tree - depth={best_dt_depth},  Test Accuracy={best_dt_acc:.4f}')


## 8. Random Forest - Overfitting Analysis

We use a **nested grid** over `max_depth` (1-8) multiply `n_estimators` (1-8) - a reduced grid suited for the 7K-row dataset.  
The seaborn heatmap shows which depth-estimator combination returning the best test accuracy.

In [ ]:
l_rf_depth = []
l_rf_est = []
l_rf_test = []
l_rf_train = []

for depth in range(1, 9):
    for estimator in range(1, 9):
        clf = RandomForestClassifier(n_estimators=estimator, max_depth=depth, random_state=1)
        clf.fit(x_train, y_train)
        l_rf_test.append(accuracy_score(y_test,  clf.predict(x_test)))
        l_rf_train.append(accuracy_score(y_train, clf.predict(x_train)))
        l_rf_depth.append(depth)
        l_rf_est.append(estimator)

rf_df = pd.DataFrame({
    'depth':          l_rf_depth,
    'n_estimators':   l_rf_est,
    'test_accuracy':  l_rf_test,
    'train_accuracy': l_rf_train
})
rf_df.head(10)

In [ ]:
best_rf_row   = rf_df.loc[rf_df['test_accuracy'].idxmax()]
best_rf_depth = int(best_rf_row['depth'])
best_rf_est   = int(best_rf_row['n_estimators'])
best_rf_acc   = best_rf_row['test_accuracy']

print(f'Best Random Forest - depth={best_rf_depth}, n_estimators={best_rf_est},  Test Accuracy={best_rf_acc:.4f}')

pivot = rf_df.pivot(index='depth', columns='n_estimators', values='test_accuracy')
plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='Oranges', linewidths=1.3)
plt.title('Random Forest Test Accuracy: Depth vs n_estimators', fontsize=14, fontweight='bold')
plt.xlabel('n_estimators', fontsize=12)
plt.ylabel('max_depth', fontsize=12)
plt.tight_layout()
plt.show()

## 9. KNN - Overfitting Analysis (Without Scaling)

We vary `k` from 1 to 30.  
At k=1 the model memorises training data (perfect train accuracy, poor test accuracy).  
As k grows the model generalises better but can eventually underfit.

In [ ]:
l_k = []
l_knn_test = []
l_knn_train = []

for k in range(1, 31):
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(x_train, y_train)
    l_knn_test.append(accuracy_score(y_test,  clf.predict(x_test)))
    l_knn_train.append(accuracy_score(y_train, clf.predict(x_train)))
    l_k.append(k)

best_knn_k   = l_k[l_knn_test.index(max(l_knn_test))]
best_knn_acc = max(l_knn_test)

plt.figure(figsize=(10, 5))
plt.plot(l_k, l_knn_train, marker='o', label='Train Accuracy', color='royalblue')
plt.plot(l_k, l_knn_test,  marker='s', label='Test Accuracy',  color='tomato')
plt.axvline(best_knn_k, color='gray', linestyle='--', alpha=0.5, label=f'Best k={best_knn_k}')
plt.xlabel('k (n_neighbors)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('KNN - Overfitting: Train vs Test Accuracy (No Scaling)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Best KNN (unscaled) - k={best_knn_k},  Test Accuracy={best_knn_acc:.4f}')

## 10. KNN - With StandardScaler

KNN is sensitive to feature scale: features like `MonthlyCharges` (0-100) dominate over binary features (0/1) without normalisation.  
`StandardScaler` transforms each feature to zero mean and unit variance, giving equal weight to all features.

In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled  = scaler.transform(x_test)

print('Scaled x_train sample (first 2 rows):')
print(x_train_scaled[:2])

In [ ]:
l_k_s = []
l_knn_s_test = []
l_knn_s_train = []

for k in range(1, 31):
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(x_train_scaled, y_train)
    l_knn_s_test.append(accuracy_score(y_test,  clf.predict(x_test_scaled)))
    l_knn_s_train.append(accuracy_score(y_train, clf.predict(x_train_scaled)))
    l_k_s.append(k)

best_knn_s_k   = l_k_s[l_knn_s_test.index(max(l_knn_s_test))]
best_knn_s_acc = max(l_knn_s_test)

plt.figure(figsize=(10, 5))
plt.plot(l_k_s, l_knn_s_train, marker='o', label='Train Accuracy', color='royalblue')
plt.plot(l_k_s, l_knn_s_test,  marker='s', label='Test Accuracy',  color='tomato')
plt.axvline(best_knn_s_k, color='gray', linestyle='--', alpha=0.5, label=f'Best k={best_knn_s_k}')
plt.xlabel('k (n_neighbors)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('KNN - Overfitting: Train vs Test Accuracy (With Scaling)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Best KNN (scaled) - k={best_knn_s_k},  Test Accuracy={best_knn_s_acc:.4f}')

## 11. Best Accuracy Comparison - All Algorithms

Collecting the best test accuracy achieved by each algorithm across their hyperparameter search  
and comparing them side by side.

In [ ]:
best_results = {
    'Decision Tree\n(best depth)':         best_dt_acc,
    'Random Forest\n(best depth & trees)': best_rf_acc,
    'KNN\n(no scaling)':                   best_knn_acc,
    'KNN\n(with scaling)':                 best_knn_s_acc
}

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

plt.figure(figsize=(10, 6))
bars = plt.bar(best_results.keys(), best_results.values(), color=colors, edgecolor='black', width=0.5)

for bar, val in zip(bars, best_results.values()):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
             f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.ylim(0.5, 1.0)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Best Test Accuracy - Algorithm Comparison', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
# Hyperparameter Tuning

Having identified **Random Forest** as our top performer, we now systematically search for optimal hyperparameters  
using cross-validated grid and random search.  
`cv=5` means 5-fold cross-validation is applied on the **training set** - the test set is only used for final evaluation.

## 12. GridSearchCV - Random Forest

`GridSearchCV` exhaustively evaluates every combination in the parameter grid.  
We search over `n_estimators` in {50, 100, 200} and `max_depth` in {None, 5, 10, 20} - 12 combinations x 5 folds = 60 fits.

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 5, 10, 20]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=1),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(x_train, y_train)

best_grid_params = grid_search.best_params_
best_grid_model  = grid_search.best_estimator_
y_pred_grid      = best_grid_model.predict(x_test)
grid_acc         = accuracy_score(y_test, y_pred_grid)

print(f'Best Parameters (GridSearchCV):  {best_grid_params}')
print(f'GridSearchCV - Test Accuracy:    {grid_acc:.4f}')

## 13. RandomizedSearchCV - Random Forest

`RandomizedSearchCV` samples `n_iter` random combinations from the parameter space - faster than GridSearch when the space is large.  
We use `n_iter=5` with `cv=5` folds for a quick comparison.

In [ ]:
param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 5, 10, 20]
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=1),
    param_distributions=param_dist,
    n_iter=5,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)
random_search.fit(x_train, y_train)

best_rand_params = random_search.best_params_
best_rand_model  = random_search.best_estimator_
y_pred_rand      = best_rand_model.predict(x_test)
rand_acc         = accuracy_score(y_test, y_pred_rand)

print(f'Best Parameters (RandomizedSearchCV): {best_rand_params}')
print(f'RandomizedSearchCV - Test Accuracy:   {rand_acc:.4f}')

## 14. Confusion Matrix - Best Tuned Model

The confusion matrix breaks down predictions into:  
- **True Positives** - correctly predicted churners  
- **True Negatives** - correctly predicted non-churners  
- **False Positives** - non-churners incorrectly flagged  
- **False Negatives** - missed churners (most costly for the business)

We automatically select the better of the two tuned models.

In [ ]:
if grid_acc >= rand_acc:
    best_final_model = best_grid_model
    best_final_pred  = y_pred_grid
    best_final_acc   = grid_acc
    method_name      = 'GridSearchCV'
else:
    best_final_model = best_rand_model
    best_final_pred  = y_pred_rand
    best_final_acc   = rand_acc
    method_name      = 'RandomizedSearchCV'

cm = confusion_matrix(y_test, best_final_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='coolwarm',
    linewidths=2,
    linecolor='white',
    annot_kws={'size': 24, 'weight': 'bold'},
    xticklabels=['No Churn', 'Churn'],
    yticklabels=['No Churn', 'Churn'],
    ax=ax
)

ax.set_xlabel('Predicted Label', fontsize=13, fontweight='bold', labelpad=10)
ax.set_ylabel('True Label',      fontsize=13, fontweight='bold', labelpad=10)
ax.tick_params(axis='both', labelsize=12)
ax.set_title(f'Confusion Matrix — Best Tuned Random Forest ({method_name})\nTest Accuracy = {best_final_acc:.4f}',fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (No Churn correctly predicted): {tn}')
print(f'False Positives (No Churn predicted as Churn):  {fp}')
print(f'False Negatives (Churn missed):                 {fn}')
print(f'True Positives  (Churn correctly predicted):    {tp}')

---
# Business & Technical Insights

## Key Findings

### Best Model
**Random Forest** achieved the highest test accuracy after tuning, outperforming both Decision Tree and KNN.  
Its ensemble structure reduces variance and captures non-linear interactions between features.

### Top Churn Drivers (from Feature Importance)
1. **Contract type** - Month-to-month customers churn at a far higher rate than 1- or 2-year contract holders
2. **Tenure** - New customers (< 12 months) are most at risk; long-term loyalty strongly reduces churn
3. **Monthly Charges** - High charges correlate with churn, especially for fiber optic internet users

These drivers align exactly with the EDA hypotheses, confirming the model is learning meaningful patterns.

### Algorithm Comparison Summary

| Algorithm | Strategy | Notes |
|-----------|----------|-------|
| Decision Tree | Rule-based splits | Interpretable; prone to overfitting at high depth |
| Random Forest | Ensemble of trees | Best overall; robust via averaging |
| KNN (no scaling) | Distance-based | Weaker; scale-sensitive features distort distances |
| KNN (with scaling) | Distance-based + StandardScaler | Improved after normalisation |

### Business Recommendation
Target **month-to-month customers** in their **first 12 months** with high monthly bills:  
- Offer loyalty discounts or contract upgrade incentives  
- Prioritise tech support for fiber optic users (high charges + high churn)  
- Use the trained model to score existing customers and flag high-risk segments for proactive retention

### Model Limitations
- **Class imbalance** (~26% churn): the model may under-predict churners
- **Static dataset**: churn drivers may shift over time - retrain periodically on fresh data